In [ ]:
#Golden Set - How I made the questions and found the pages

#File: Data/GoldenSet_Questions.csv (no header row)
#Format: id, "question", "type", True/False, "locations"
#Unanswerable questions just have an empty location

#40 questions total: 16 single, 7 comparison, 7 distractor, 5 multi-hop, 5 unanswerable

#Where the questions came from
#Claude helped draft some of them, I reviewed all of them

#Tool 1: where() which just searches the text
#Prints every doc that contains the exact words, with a snippet around them
#Used it for Core Rules: search the rule name, read the page, pick the one that actually DEFINES the rule
#The page number printed on the rulebook matches the id (page 29 = core-rules--p029)
def where(term, width=120):
    term_lower = term.lower()
    for doc in docs:
        text = " ".join(doc["text"].split())          #flatten line breaks so the snippet reads clean
        position = text.lower().find(term_lower)
        if position >= 0:
            print(f"{doc['id']:40} ...{text[max(0, position - 40):position + width]}...")

#Ex: where("lethal hits")


#Tool 2: candidates() uses vector search, for when the exact words don't match
#Used it for Lore -- where() kept missing stuff (the wiki says "kill-team" not "Kill Team")
#Rules I followed so it stays honest:
def candidates(probe, k=15):
    seen_doc_ids = set()
    for chunk in retrieve(probe, k):
        if chunk["doc_id"] not in seen_doc_ids:     
            seen_doc_ids.add(chunk["doc_id"])
            snippet = " ".join(chunk["text"].split())[:120]
            print(f"{chunk['vector_score']:.3f}  {chunk['doc_id']:40} {snippet}")

#Ex: candidates("how Deathwatch kill teams are formed and fight")

#Datasheets didn't need a tool the id is just the file name in corpus/datasheets
#Labels are document ids, not chunk ids -

#Known limits
#Some lore pages were found with the retrieval system itself, so lore recall might be a little generous
#Open ended questions can have more right answers than I labelled
#Ex: the "black ops close combat" question pulled the Deathwatch Assault Marines page instead of mine
#it happily recommended units that don't even have a datasheet in my corpus
#No reference answers in the CSV, so the judge only grades against the retrieved chunks